# Phase 1b — controls and attacker ladder

Append these cells to the **end** of `phase1_ntu60_identity_leakage.ipynb`,
after the Table 1 / summary cell. They reuse the objects already in memory
(`RESULTS`, `SUBJ_LABEL`, `ACT_LABEL`, `SPLITS`, `CKPT_DIR`, `RES_DIR`,
`run_experiment`), so **run them in the same session** as Phase 1.

If the kernel has restarted, re-run Phase 1 cells 1–7 first (loading and
caching only — no training), then come back here.

Purpose of each cell:

| Cell | Answers the reviewer question |
|---|---|
| B0 | — adapter, discovers the model class and data layout |
| B1 | extracts frozen 256-d features once, reused by B2–B4 |
| B2 | "is 12.19% a real ceiling or just bad SGD?" |
| B3 | "does the backbone leak more about people it was trained on?" |
| B4 | "what happens if the attacker tries harder?" |
| B5 | assembles Table 2 and the ladder figure |
| B6 | seeds 2 and 3 — run last, it is the expensive one |

## Cell B0 — adapter

Finds the pieces Phase 1 left in memory. Prints what it found so you can
check it picked the right things. Nothing is trained here.

In [ ]:
import os, glob, json, inspect, time
import numpy as np
import torch
import torch.nn as nn

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- 1. the cached skeleton array -------------------------------------
def _find_array():
    for name in ['X', 'DATA', 'SKEL', 'X_ALL', 'x_all', 'ARR', 'XDATA']:
        v = globals().get(name)
        if isinstance(v, np.ndarray) and v.ndim == 5:
            print('array in memory: `%s` %s %s' % (name, v.shape, v.dtype))
            return v
    dirs = [v for k, v in globals().items()
            if k.endswith('_DIR') and isinstance(v, str) and os.path.isdir(v)]
    dirs += ['/content', '/kaggle/working', '/kaggle/temp', '.']
    seen = set()
    for d in dirs:
        if d in seen:
            continue
        seen.add(d)
        hits = glob.glob(os.path.join(d, '**', '*T64_x.npy'), recursive=True)
        if hits:
            print('loading cache from disk:', hits[0])
            return np.load(hits[0], mmap_mode='r')
    raise RuntimeError('skeleton cache not found. Set X_ALL manually, e.g. '
                       'X_ALL = np.load("/content/ntu60_T64_x.npy", mmap_mode="r")')

X_ALL = _find_array()

# ---- 2. the model class ------------------------------------------------
_model_cls = None
for _n, _v in globals().items():
    if inspect.isclass(_v) and hasattr(_v, 'extract'):
        _model_cls = _v
        print('model class:', _n)
        break
if _model_cls is None:
    raise RuntimeError('no class with an .extract() method found — '
                       're-run the ST-GCN definition cell')

def build_model(num_classes):
    """Instantiate the Phase 1 backbone, tolerating either kwarg spelling."""
    sig = inspect.signature(_model_cls.__init__).parameters
    key = 'num_classes' if 'num_classes' in sig else 'num_class'
    return _model_cls(**{key: num_classes})

# ---- 3. the checkpoint written by Experiment A -------------------------
A_CKPT = RESULTS['A_action_xsub']['ckpt']
if not os.path.exists(A_CKPT):
    hits = glob.glob(os.path.join(CKPT_DIR, 'A_action_xsub_best.pt'))
    A_CKPT = hits[0]
print('experiment A checkpoint:', A_CKPT)

def load_backbone(num_classes=60):
    m = build_model(num_classes)
    ck = torch.load(A_CKPT, map_location='cpu')
    sd = ck.get('model', ck.get('state_dict', ck))
    missing, unexpected = m.load_state_dict(sd, strict=False)
    if missing:
        print('  note: %d missing keys (expected if head differs)' % len(missing))
    return m.to(DEV).eval()

# ---- 4. work out which tensor layout .extract() wants ------------------
_LAYOUT = None
def _as_input(batch):
    """batch: numpy slice of X_ALL -> tensor in the layout the model accepts."""
    global _LAYOUT
    t = torch.from_numpy(np.ascontiguousarray(batch)).float()
    if _LAYOUT is not None:
        return t.permute(*_LAYOUT).contiguous() if _LAYOUT != 'id' else t
    probe = load_backbone(60)
    for cand in ['id', (0, 4, 2, 3, 1), (0, 3, 2, 1, 4), (0, 2, 1, 3, 4)]:
        try:
            tt = t if cand == 'id' else t.permute(*cand).contiguous()
            with torch.no_grad():
                probe.extract(tt[:2].to(DEV))
            _LAYOUT = cand
            print('tensor layout:', cand)
            del probe
            return tt
        except Exception:
            continue
    raise RuntimeError('could not find a working input layout for .extract()')

# ---- 5. splits ---------------------------------------------------------
def get_split(name):
    s = SPLITS[name]
    if isinstance(s, dict):
        return np.asarray(s['train']), np.asarray(s['test'])
    return np.asarray(s[0]), np.asarray(s[1])

_tr, _te = get_split('reid_rand')
print('reid_rand split: %d train / %d test' % (len(_tr), len(_te)))
print('device:', DEV)

## Cell B1 — extract the frozen features once

Runs the Experiment A backbone over every clip in the `reid_rand` split and
stores the 256-d vector it produces. Cells B2–B4 then work on these vectors
instead of re-running the network, which is why they take minutes rather
than hours.

No training happens here. Expect 2–4 minutes.

In [ ]:
FEAT_PATH = os.path.join(RES_DIR, 'A_features_reid_rand.npz')

@torch.no_grad()
def extract_features(idx, bs=256):
    model = load_backbone(60)
    out = []
    for i in range(0, len(idx), bs):
        chunk = X_ALL[np.sort(idx[i:i + bs])]
        f = model.extract(_as_input(chunk).to(DEV))
        if f.ndim > 2:                      # pool anything left over
            f = f.mean(dim=tuple(range(2, f.ndim)))
        out.append(f.float().cpu().numpy())
        if i % (bs * 20) == 0:
            print('  %d / %d' % (i, len(idx)), end='\r')
    del model
    torch.cuda.empty_cache()
    return np.concatenate(out, 0)

if os.path.exists(FEAT_PATH):
    z = np.load(FEAT_PATH)
    F_tr, F_te, y_tr, y_te = z['F_tr'], z['F_te'], z['y_tr'], z['y_te']
    print('loaded cached features')
else:
    tr_idx, te_idx = get_split('reid_rand')
    tr_idx, te_idx = np.sort(tr_idx), np.sort(te_idx)
    t0 = time.time()
    F_tr = extract_features(tr_idx)
    F_te = extract_features(te_idx)
    y_tr = np.asarray(SUBJ_LABEL)[tr_idx]
    y_te = np.asarray(SUBJ_LABEL)[te_idx]
    np.savez_compressed(FEAT_PATH, F_tr=F_tr, F_te=F_te, y_tr=y_tr, y_te=y_te)
    print('extracted in %.0f s' % (time.time() - t0))

print('train features', F_tr.shape, ' test features', F_te.shape)
print('identity classes present:', len(np.unique(y_tr)))

## Cell B2 — the linear ceiling

Experiment C used SGD with a learning rate and a schedule, so a sceptic can
argue that 12.19% reflects the optimiser rather than the features.

Logistic regression with L-BFGS removes that argument: it is convex, so it
converges to *the* optimum, not *an* optimum. Whatever it returns is the true
linear separability of identity in the action representation.

- returns ≈12% → Experiment C was correct, the finding stands
- returns 25%+ → the SGD probe was under-trained and the headline changes

CPU only, 3–8 minutes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

sc = StandardScaler().fit(F_tr)
Z_tr, Z_te = sc.transform(F_tr), sc.transform(F_te)

LADDER = {}

t0 = time.time()
lr_clf = LogisticRegression(max_iter=3000, C=1.0, n_jobs=-1)
lr_clf.fit(Z_tr, y_tr)
acc_lin = 100 * lr_clf.score(Z_te, y_te)
LADDER['linear (logistic regression)'] = acc_lin
print('linear ceiling      : %.2f%%   (%.0f s)' % (acc_lin, time.time() - t0))

t0 = time.time()
mlp = MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=300,
                    early_stopping=True, n_iter_no_change=15, random_state=1)
mlp.fit(Z_tr, y_tr)
acc_mlp = 100 * mlp.score(Z_te, y_te)
LADDER['MLP probe (512-256)'] = acc_mlp
print('non-linear probe    : %.2f%%   (%.0f s)' % (acc_mlp, time.time() - t0))

print('\nExperiment C (SGD linear probe) was %.2f%%' %
      RESULTS['C_probe_frozen']['best_acc'])
print('difference vs convex optimum: %+.2f points' %
      (acc_lin - RESULTS['C_probe_frozen']['best_acc']))

## Cell B3 — seen vs unseen subjects

The Experiment A backbone was trained on the X-Sub split, so it saw only 20
of the 40 people. The probe in Experiment C was then asked about all 40.

That mixes two different privacy questions:

- how much does the model remember about people **in its training data**?
- how much does it leak about **arbitrary** people it has never seen?

The second is the question that matters for deployment, but you must report
both, and the gap between them is itself a result. Note the chance levels
differ (5.0% for 20 classes, 2.5% for 40).

CPU only, 2–5 minutes.

In [ ]:
XS_TRAIN = globals().get('XSUB_TRAIN',
    [1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38])
seen_lbl = set(s - 1 for s in XS_TRAIN)          # SUBJ_LABEL is 0-indexed

def probe_subset(mask_tr, mask_te, name, chance):
    if mask_tr.sum() < 50 or mask_te.sum() < 50:
        print('%-22s: too few samples, skipped' % name)
        return None
    clf = LogisticRegression(max_iter=3000, n_jobs=-1)
    clf.fit(Z_tr[mask_tr], y_tr[mask_tr])
    a = 100 * clf.score(Z_te[mask_te], y_te[mask_te])
    print('%-22s: %6.2f%%   chance %.2f%%   %5.1fx' % (name, a, chance, a / chance))
    return a

m_seen_tr = np.isin(y_tr, list(seen_lbl))
m_seen_te = np.isin(y_te, list(seen_lbl))

SEEN = {}
SEEN['seen (20 subj)']   = probe_subset(m_seen_tr,  m_seen_te,  'seen subjects',   5.0)
SEEN['unseen (20 subj)'] = probe_subset(~m_seen_tr, ~m_seen_te, 'unseen subjects', 5.0)
SEEN['all (40 subj)']    = probe_subset(np.ones(len(y_tr), bool),
                                        np.ones(len(y_te), bool), 'all subjects', 2.5)

## Cell B4 — the attacker ladder

This is the methodological contribution. Existing papers report leakage as a
single number against a single attacker, which makes a privacy claim
impossible to falsify. Reporting it as a curve over attacker capacity does not.

You already own both ends: 12.19% (weakest) and 78.86% (strongest). This cell
fills the middle rung — an attacker who starts from the action model and
fine-tunes the whole network for identity.

**This one trains.** Roughly 45 minutes. It should land between the MLP probe
and the from-scratch attacker; if it lands *above* 78.86%, that is itself
worth reporting, because it means action pre-training actively helps an
adversary.

In [ ]:
RESULTS['D_finetune_from_A'] = run_experiment(
    tag='D_finetune_from_A', labels=SUBJ_LABEL, split='reid_rand',
    num_classes=40, epochs=80, lr=0.05, bs=64,
    init_from=A_CKPT, freeze=False)
save_results()

## Cell B5 — Table 2 and the ladder figure

Writes `results/table2_ladder.txt`, `results/phase1b.json` and
`results/attacker_ladder.png`.

In [ ]:
import matplotlib.pyplot as plt

LADDER_FULL = [
    ('Linear probe, SGD (Exp. C)',    RESULTS['C_probe_frozen']['best_acc']),
    ('Linear probe, convex optimum',  LADDER.get('linear (logistic regression)')),
    ('Non-linear MLP probe',          LADDER.get('MLP probe (512-256)')),
    ('Full fine-tune from action model',
     RESULTS.get('D_finetune_from_A', {}).get('best_acc')),
    ('Dedicated re-ID, from scratch (Exp. B)',
     RESULTS['B_reid_rand']['best_acc']),
]
LADDER_FULL = [(n, a) for n, a in LADDER_FULL if a is not None]

lines = ['Table 2. Identity leakage as a function of attacker capacity',
         '(NTU RGB+D 60, 40 identities, random 80/20 split, chance = 2.50%)',
         '',
         '%-42s %8s %9s' % ('Attacker', 'Top-1', 'x chance'),
         '-' * 62]
for n, a in LADDER_FULL:
    lines.append('%-42s %7.2f%% %8.1fx' % (n, a, a / 2.5))

lines += ['', 'Seen vs unseen subjects (linear probe on frozen features)', '']
for k, v in SEEN.items():
    if v is not None:
        ch = 5.0 if '20' in k else 2.5
        lines.append('%-42s %7.2f%% %8.1fx' % (k, v, v / ch))

txt = '\n'.join(lines)
print(txt)
open(os.path.join(RES_DIR, 'table2_ladder.txt'), 'w').write(txt)
json.dump({'ladder': LADDER_FULL, 'seen_unseen': SEEN},
          open(os.path.join(RES_DIR, 'phase1b.json'), 'w'), indent=2)

fig, ax = plt.subplots(figsize=(8, 4.5))
names = [n for n, _ in LADDER_FULL]
vals = [a for _, a in LADDER_FULL]
ax.barh(range(len(vals)), vals)
ax.set_yticks(range(len(vals)))
ax.set_yticklabels(names, fontsize=8)
ax.invert_yaxis()
ax.axvline(2.5, ls='--', c='r', label='chance (2.5%)')
ax.set_xlabel('re-identification accuracy (%)')
ax.set_title('Identity leakage vs attacker capacity')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RES_DIR, 'attacker_ladder.png'), dpi=150)
plt.show()

## Cell B6 — seeds 2 and 3 (run last)

A Q1 reviewer will not accept single-seed numbers. This repeats the three
headline runs at two further seeds so you can report mean ± SD.

Note the data splits stay fixed — only initialisation and augmentation vary.
State that explicitly in the paper; it isolates optimisation variance, which
is the quantity readers care about here.

**This is the expensive cell: roughly 4 hours.** Run it overnight, or after
you have Colab Pro or cluster access. Everything else in Phase 1b should be
done first.

In [ ]:
import random

SEED_RESULTS = {}
for seed in [2, 3]:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    SEED_RESULTS['A_s%d' % seed] = run_experiment(
        tag='A_action_xsub_s%d' % seed, labels=ACT_LABEL, split='xsub',
        num_classes=60, epochs=60, lr=0.1, bs=64)

    SEED_RESULTS['B_s%d' % seed] = run_experiment(
        tag='B_reid_rand_s%d' % seed, labels=SUBJ_LABEL, split='reid_rand',
        num_classes=40, epochs=80, lr=0.1, bs=64)

    SEED_RESULTS['C_s%d' % seed] = run_experiment(
        tag='C_probe_frozen_s%d' % seed, labels=SUBJ_LABEL, split='reid_rand',
        num_classes=40, epochs=80, lr=0.05, bs=256,
        init_from=A_CKPT, freeze=True)
    save_results()

for grp, base in [('A', 'A_action_xsub'), ('B', 'B_reid_rand'), ('C', 'C_probe_frozen')]:
    vals = [RESULTS[base]['best_acc']]
    vals += [SEED_RESULTS[k]['best_acc'] for k in SEED_RESULTS if k.startswith(grp + '_')]
    print('%-16s %.2f +/- %.2f  (n=%d)' % (base, np.mean(vals), np.std(vals), len(vals)))